In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Colab Notebooks/AI/'

# 3. 허깅페이스 활용

- 다양한 전문가들이 거대 데이터로 오랜시간 학습시킨 `사전 학습된 모델`의 저장고(hub)
- 사전 학습된 모델(Pre-Trained Model)을 활용한 전이 학습(Transfer Learning) 과정에 대한 상세한 내용은 후반부 챕터에서 더 다룰 예정
- 지금은 허깅페이스에서 지정된 모델을 불러와 학습시키고, 우리가 만든 임베딩 층에 필요한 정보를 이식 시키는 작업을 진행

## 3-1. 사전 준비

1. 데이터는 챕터 1에서 처리된 `nsmc.txt` 사용
2. 토크나이저도 챕터 1에서 생성한 tokenizer 그대로 활용
3. 임베딩 레이어는 챕터 2에서 생성한 embedding_layer 사용 (무작위 가중치 행렬)

In [ ]:
import torch
import torch.nn as nn
from tokenizers import BertWordPieceTokenizer


vocab_size = 30000     
tokenizer = BertWordPieceTokenizer(lowercase=False, strip_accents=False)

# 1 ~ 2. 사전 처리된 데이터로 토크나이저 학습
tokenizer.train(
    # files='nsmc.txt',
    files=base_path + 'nsmc.txt',
    vocab_size=vocab_size,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
)

# 3. 임베딩
embedding_dim = 768    
# 무작위 가중치 행렬로 초기화된 임베딩 레이어 생성
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

## 3-2. 사전 학습 모델 로드

- 허깅페이스를 통해, 한국어 데이터가 학습된 모델 불러오기

### 3-2-1. 허깅페이스에서 모델 찾기 가이드

- 허깅페이스에는 사전 훈련 모델이 매우 방대하게 공유되고 있음
- 라이브러리를 통해서 원하는 모델을 불러오기 전, 플랫폼에서 직접 원하는 모델을 검색하고 선택할 수 있어야 함.

1. 모델 검색 및 필터링
- 허깅페이스 [Models 페이지](https://huggingface.co/models)에서 검색할 때, 다양한 기준으로 검색 해 봅시다.

| 검색 기준 | 설명 | 필터링 예 |
| --- | --- | --- |
| 태스크 | 모델이 수행하는 처리 유형에 따라서 필터링 | text-classification (문장 분류), question-answering (질의응답), translation (번역) |
| 언어  | 모델이 지원하는 언어를 기준으로 필터링 | ko (한국어), en (영어), zh (중국어) 등 |
| 라이브러리 | 모델이 구현된 주요 라이브러리를 기준으로 필터링 | PyTorch, scikit, TensorFlow, JAX 등 |
| 모델 이름 | 특정 모델 이름, 저자, 또는 기술적 특징 등 | bert, gpt, google  |

### 3-2-2. 사전 학습된 모델 불러오기

- 이번 실습에서는 미리 찾아 둔, `klue/bert-base` 모델을 사용 할 것
    - 키워드: `klue`, `Transformers`, `PyTorch`, `Korean`, `bert`
    - 선정 기준
        - 토크나이징을 BERT 기반으로 하였으므로, BERT 기반의 모델
        - 임베딩을 PyTorch를 활용하고 있으므로 PyTorch 기반의 모델
        - 한국 영화 리뷰 텍스트를 처리하고 있으므로 Korean 기반의 모델

In [ ]:
# transformers 라이브러리에서 제공하는 사전 학습된 모델과 토크나이저를 불러오기 위한 모듈
    # AutoTokenizer: 다양한 모델에 맞는 토크나이저를 자동으로 불러오는 클래스
    # AutoModel: 다양한 모델에 맞는 사전 학습된 모델을 자동으로 불러오는 클래스
from transformers import AutoTokenizer, AutoModel

# (Pre-trained Model) 로드 -> klue/bert-base
pretrained_model_name = "klue/bert-base"
# from_pretrained 메서드는 모델 이름을 입력받아 해당 모델에 맞는 토크나이저를 자동으로 불러옴
pretrained_tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name)
pretrained_model = AutoModel.from_pretrained(pretrained_model_name)

# klue/bert-base의 임베딩 레이어에서 가중치 행렬(임베딩 벡터들)을 추출
pretrained_embeddings = pretrained_model.embeddings.word_embeddings.weight

# 장치 설정 (GPU 사용 가능 시 GPU 사용)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 디바이스 이동
embedding_layer = embedding_layer.to(device)
pretrained_embeddings = pretrained_embeddings.to(device)

## 3-3. 지식 이전

- 무작위 가중치 행렬로 이뤄진 `embedding_layer`에 잘 학습된 임베딩 정보가 담긴 `pretrained_embeddings`의 내용을 옮겨 담을 것.
- 단, 무작위로 옮겨 담는 것이 아닌, 우리 단어 사전에 있는 단어가 학습된 임베딩 정보에도 있을때, 해당 데이터를 복사해 올 것

In [ ]:
from tqdm import tqdm # 진행 상황을 시각적으로 보여주는 라이브러리

# get_vocab(): 토크나이저의 단어 사전을 딕셔너리 형태로 반환
custom_vocab = tokenizer.get_vocab()

# 공통 단어에 대해 Pre-trained 벡터를 복사
copied_count = 0
# no_grad(): 이 블록 내에서는 그래디언트 계산을 하지 않아 메모리를 효율적으로 사용
with torch.no_grad(): 
    # custom_vocab의 각 단어에 대해 반복
    # tqdm: 진행 상황을 시각적으로 보여줌
    for token, token_id in tqdm(custom_vocab.items()):
        # klue/bert-base의 토크나이저는 BertWordPieceTokenizer와 동일한 구조를 사용
        # 해당 단어가 사전 학습된 토크나이저의 단어 사전에 존재하는지 확인
        if token in pretrained_tokenizer.vocab:
            # 단어에 해당하는 ID 추출
            pretrained_id = pretrained_tokenizer.convert_tokens_to_ids(token)
            # 임베딩 레이어의 가중치 행렬에서 해당 단어의 벡터를 복사
            embedding_layer.weight[token_id] = pretrained_embeddings[pretrained_id]
            # 복사된 단어 수 증가
            copied_count += 1

print(f"총 {len(custom_vocab)}개의 단어 중 {copied_count}개의 벡터를 성공적으로 이식했습니다.")

## 3-3. 결과 확인

In [ ]:
import torch.nn.functional as F

target_word = "영화"
top_k = 5

# 기준 단어의 ID와 벡터 조회
target_id = tokenizer.token_to_id(target_word)
target_vector = embedding_layer.weight[target_id]

# 전체 단어 벡터와 코사인 유사도 계산
all_vectors = embedding_layer.weight
similarities = F.cosine_similarity(target_vector.unsqueeze(0), all_vectors, dim=1)

# 유사도가 높은 Top-K 단어 찾기 (자기 자신 제외)
top_scores, top_indices = torch.topk(similarities, k=top_k + 1)

print(f"학습 전, '{target_word}'와 가장 유사한 단어 Top {top_k}:")
for i in range(1, top_k + 1):
    similar_word_id = top_indices[i].item()
    similar_word = tokenizer.id_to_token(similar_word_id)
    score = top_scores[i].item()
    print(f"{i}순위: {similar_word} (유사도: {score:.4f})")

# 현재 결과는 유의미한 단어가 나올 수 있을까?